In [1]:
import numpy as np
import pandas as pd

from functools import partial
import importlib

import src.data
import src.features
import src.kernel_normalisation
import src.kernels
import src.krr
import src.metrics
import src.multikernel

from src.data import encode_labels, load_data, train_val_split
from src.features import HOG, HSVHistogram
from src.kernel_normalisation import unit_diag
from src.kernels import *
from src.krr import KernelRidgeRegression
from src.metrics import accuracy
from src.multikernel import *

In [2]:
DATA_DIR = "data/"

X_tr, X_te, y_tr_raw = load_data(DATA_DIR)

y_tr, inv_map = encode_labels(y_tr_raw)

n_classes = len(np.unique(y_tr))

### HSV features

In [3]:
hsv = HSVHistogram(bins=(8, 8, 8), grid=(4,4)).fit(X_tr)

X_tr_hsv = hsv.transform(X_tr)
X_te_hsv = hsv.transform(X_te)

X_train_hsv, X_val_hsv, y_train_hsv, y_val_hsv = train_val_split(
    X_tr_hsv, y_tr, test_size=0.3, seed=20
)

In [49]:
gamma = estimate_gamma(X_train_hsv)

lams = [1e-3, 1e-4, 1e-5]

best = {"acc": -1.0, "lam": None, "gamma": None}

for lam in lams:
    kernel_fn = partial(
        gaussian_kernel_matrix,
        gamma=gamma,
    )
    model = KernelRidgeRegression(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lam=lam,
    ).fit(X_train_hsv, y_train_hsv, X_val=X_val_hsv, y_val=y_val_hsv)

    pred_val, _ = model.predict(X_val_hsv)
    acc = accuracy(y_val_hsv, pred_val)
    print(f"lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lam": lam, "gamma": gamma}

print("best:", best)

lam=1.0e-03, val_acc=0.3673
lam=1.0e-04, val_acc=0.3773
lam=1.0e-05, val_acc=0.3373
best: {'acc': 0.37733333333333335, 'lam': 0.0001, 'gamma': 0.40715705765390975}


### HOG features

In [18]:
hog = HOG(
    image_shape=(3, 32, 32),
    colour_mode="rgb",
    cell_size=8,
    block_size=2,
    block_stride=1,
    num_bins=6,
    signed=False,
).fit(X_tr)

X_tr_hog = hog.transform(X_tr)
X_te_hog = hog.transform(X_te)

X_train_hog, X_val_hog, y_train_hog, y_val_hog = train_val_split(
    X_tr_hog, y_tr, test_size=0.3, seed=20
)

Grid search over kernel weights

In [50]:
gammas_hog = estimate_chi2_gammas_channel(X_train_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

gamma_hsv = estimate_gamma(X_train_hsv)
hsv_gaussian = partial(gaussian_kernel_matrix, gamma=gamma_hsv)

# Two-kernel setup:
specs: list[KernelSpec] = [
    {
        "name": "hog_chi2",
        "Z": X_train_hog,
        "kernel_fn": hog_chi2,
        "diag_fn": unit_diag,
        "normalise": True,
    },
    {
        "name": "hsv_gaussian",
        "Z": X_train_hsv,
        "kernel_fn": hsv_gaussian,
        "diag_fn": unit_diag,
        "normalise": True,
    },
]

out = beta_grid_search_cv_two_kernels(
    specs,
    y_train_hog,
    n_classes=n_classes,
    model="krr",
    betas=np.linspace(0, 1, 21),
    k=5,
    seed=20,
    lam=1e-4,
    verbose=1,
)

print("best_beta:", out["best_beta"], "best_mean_acc:", out["best_mean_acc"])

beta=0.000 mean_acc=0.3789
beta=0.050 mean_acc=0.5377
beta=0.100 mean_acc=0.5674
beta=0.150 mean_acc=0.5763
beta=0.200 mean_acc=0.5834
beta=0.250 mean_acc=0.5911
beta=0.300 mean_acc=0.5951
beta=0.350 mean_acc=0.5971
beta=0.400 mean_acc=0.5946
beta=0.450 mean_acc=0.5954
beta=0.500 mean_acc=0.5954
beta=0.550 mean_acc=0.5929
beta=0.600 mean_acc=0.5920
beta=0.650 mean_acc=0.5906
beta=0.700 mean_acc=0.5889
beta=0.750 mean_acc=0.5886
beta=0.800 mean_acc=0.5877
beta=0.850 mean_acc=0.5854
beta=0.900 mean_acc=0.5786
beta=0.950 mean_acc=0.5706
beta=1.000 mean_acc=0.5489
best_beta: 0.3499999940395355 best_mean_acc: 0.5971428571428572


## After tuning parameters, train the best model.

### Check the best model

In [52]:
beta = float(np.asarray(out["best_beta"]).item())

# IMPORTANT: use the partial kernels you defined (with gammas)
Khs_tr = hsv_gaussian(X_train_hsv, X_train_hsv)
Kh_tr = hog_chi2(X_train_hog, X_train_hog)

# Match CV: unit-diagonal normalisation per kernel
Ksh_tr = normalise_train_gram(Khs_tr, unit_diag(X_train_hsv))
Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_train_hog))

Ktr = beta * Ksh_tr + (1.0 - beta) * Kh_tr

Ks_val = hsv_gaussian(X_val_hsv, X_train_hsv)
Kh_val = hog_chi2(X_val_hog, X_train_hog)

Ks_val = normalise_cross_gram(Ks_val, unit_diag(X_val_hsv), unit_diag(X_train_hsv))
Kh_val = normalise_cross_gram(Kh_val, unit_diag(X_val_hog), unit_diag(X_train_hog))

K_val = beta * Ks_val + (1.0 - beta) * Kh_val

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_train_hsv, K=Ktr
)
yhat_val, _ = best_model.predict(K_star=K_val)
print("val acc:", accuracy(y_val_hog, yhat_val))

val acc: 0.596


### Train on full dataset for submission

In [54]:
beta = float(np.asarray(out["best_beta"]).item())

# Re-estimate gammas on FULL training set (not the CV subset)
gamma_hsv = estimate_gamma(X_tr_hsv)
hsv_gaussian = partial(gaussian_kernel_matrix, gamma=gamma_hsv)

gammas_hog = estimate_chi2_gammas_channel(X_tr_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

# FULL train Gram
Ks_tr = hsv_gaussian(X_tr_hsv, X_tr_hsv)
Kh_tr = hog_chi2(X_tr_hog, X_tr_hog)

Ks_tr = normalise_train_gram(Ks_tr, unit_diag(X_tr_hsv))
Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_tr_hog))

Ktr = beta * Ks_tr + (1.0 - beta) * Kh_tr

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_tr, K=Ktr
)

# TEST x TRAIN cross Gram
Ks_te_tr = hsv_gaussian(X_te_hsv, X_tr_hsv)
Kh_te_tr = hog_chi2(X_te_hog, X_tr_hog)

Ks_te_tr = normalise_cross_gram(Ks_te_tr, unit_diag(X_te_hsv), unit_diag(X_tr_hsv))
Kh_te_tr = normalise_cross_gram(Kh_te_tr, unit_diag(X_te_hog), unit_diag(X_tr_hog))

K_star = beta * Ks_te_tr + (1.0 - beta) * Kh_te_tr

yte_int, _ = best_model.predict(K_star=K_star)
yte = np.array([inv_map[i] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
sub.to_csv("results/submission.csv", index_label="Id")